# Care Plan Agent and Plan Verifier (owner: Mary)

Experiments only: production code imports `backend/app/agents/care_plan.py`, `plan_verifier.py` and `plan_rules.py`.
Run from the repository root with the `.venv` that has `requirements.txt` installed. The model cells need
`GROQ_API_KEY` in `.env`. Clear all outputs before committing.

In [ ]:
import json, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "backend"))
load_dotenv(ROOT / ".env")

from app.agents import care_plan, plan_rules, plan_verifier
from app.models import PlanDraft

SAMPLES = ROOT / "data" / "sample_discharges"
EXPECTED = {k: v for k, v in json.loads((SAMPLES / "expected.json").read_text(encoding="utf-8")).items() if not k.startswith("_")}
list(EXPECTED)

## 1. Rule checks only (no model)

Each note's reference plan against its note. These must match `expected.json`.

In [ ]:
for name, case in EXPECTED.items():
    note = (SAMPLES / name).read_text(encoding="utf-8")
    print(f"== {name}: {case['seeded']}")
    for issue in plan_rules.issues(note, PlanDraft.model_validate(case["plan"])):
        print(f"   {issue.severity:6} {issue.field:24} {issue.issue}  [{issue.source_text}]")

## 2. Try a seeded error

Change a value the note does not support and see the verifier block delivery.

In [ ]:
import copy
plan = copy.deepcopy(EXPECTED["tola-adeyemi.txt"]["plan"])
plan["medications"][0]["dose"] = "2 tablets"   # the note says one tablet
note = (SAMPLES / "tola-adeyemi.txt").read_text(encoding="utf-8")
await plan_verifier.run({"source_note": note, "plan": plan, "use_model": False})

## 3. Full run with Groq

Draft each note with the Care Plan Agent, then verify the draft.

In [ ]:
for name, case in EXPECTED.items():
    note = (SAMPLES / name).read_text(encoding="utf-8")
    drafted = await care_plan.run({"source_note": note})
    checked = await plan_verifier.run({"source_note": note, "plan": drafted["plan"]})
    print(f"== {name}  deliverable={drafted['deliverable']}  verdict={checked['verdict']}  model_checked={checked['model_checked']}")
    for issue in checked["issues"]:
        print(f"   {issue['severity']:6} {issue['field']:24} {issue['issue']}")